## Import the important libraries

In [91]:
import pandas as pd
import numpy as np
import re

# imports for data visualisation
import seaborn as sns
from PIL import Image
from scipy import stats
import matplotlib.pyplot as plt
from nltk.probability import FreqDist
from wordcloud import WordCloud, ImageColorGenerator
%matplotlib inline

# imports for pre-processing
import os
from nltk.tokenize import WordPunctTokenizer
from bs4 import BeautifulSoup
from collections import Counter
from nltk.corpus import wordnet
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tokenize import TweetTokenizer
from nltk.stem.porter import *
import string
import nltk

## Read in the data

In [103]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test_with_no_labels.csv")

# Create a copy for EDA
train_eda = df_train.copy()

# Create copies for modeling
train = df_train.copy()
test = df_test.copy()

In [104]:
df_train.head()

,sentiment,message,tweetid
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221
1,1,It's not like we lack evidence of anthropogeni...,126103
2,2,RT @RawStory: Researchers say we have three ye...,698562
3,1,#TodayinMaker# WIRED : 2016 was a pivotal year...,573736
4,1,"RT @SoyNovioDeTodas: It's 2016, and a racist, ...",466954


In [105]:
df_test.head()

,message,tweetid
0,Europe will now be looking to China to make su...,169760
1,Combine this with the polling of staffers re c...,35326
2,"The scary, unimpeachable evidence that climate...",224985
3,@Karoli @morgfair @OsborneInk @dailykos \r\nPu...,476263
4,RT @FakeWillMoore: 'Female orgasms cause globa...,872928


In [106]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15819 entries, 0 to 15818
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  15819 non-null  int64 
 1   message    15819 non-null  object
 2   tweetid    15819 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 370.9+ KB


Checking if there are missing values...


In [107]:
df_train.isna().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15819 entries, 0 to 15818
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   sentiment  15819 non-null  bool 
 1   message    15819 non-null  bool 
 2   tweetid    15819 non-null  bool 
dtypes: bool(3)
memory usage: 46.5 KB


In [108]:
df_train.isnull().sum()

sentiment    0
message      0
tweetid      0
dtype: int64

There are no missing values in the train dataset.

## Pre-processing

First, we extract mentions and hashtags

In [109]:
def get_hashtags_and_mentions(df):
    df["hashtags"] = df["message"].str.lower().str.findall(r'#.*?(?=\s|$)')
    hashtags = df["hashtags"]
    
    df["mentions"] = df["message"].str.lower().str.findall(r'@\w*')
    mentions = df["mentions"]
    
    df["url"] = df["message"].str.lower().str.findall(r'http\S+|www.\S+')
    urls = df["url"]

    df["hashtags"] = [np.nan if len(x) == 0 else x for x in hashtags]
    df["mentions"] = [np.nan if len(x) == 0 else x for x in mentions]
    df["url"] = [np.nan if len(x) == 0 else x for x in urls]
    
    return df

In [110]:
get_hashtags_and_mentions(df_train).head()

,sentiment,message,tweetid,hashtags,mentions,url
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc]
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN
2,2,RT @RawStory: Researchers say we have three ye...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]"
3,1,#TodayinMaker# WIRED : 2016 was a pivotal year...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd]
4,1,"RT @SoyNovioDeTodas: It's 2016, and a racist, ...",466954,[#electionnight],[@soynoviodetodas],NaN


Second, we remove Twitter handles (e.g., @RawStory), hashtags, mentions and urls. These do not provide any important information related to our objective

In [111]:
def remove_urls(text):
    return re.sub(r'http\S+|www.\S+', '', text)

def remove_twitter_handles(text):
    return re.sub(r'@\w*', '', text)

def remove_hashtags(text):
    return re.sub(r'#.*?(?=\s|$)', '', text)

def remove_rt(text):
    return text.replace('RT', '')

df_train["message"] = df_train["message"].apply(remove_urls)
df_train["message"] = df_train["message"].apply(remove_twitter_handles)
df_train["message"] = df_train["message"].apply(remove_hashtags)
df_train["message"] = df_train["message"].apply(remove_rt)

df_train.head()

,sentiment,message,tweetid,hashtags,mentions,url
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc]
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]"
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd]
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN


## Tokenization

Tokenization is the process of tokenizing or splitting a string, text into a list of tokens. One can think of token as parts like a word is a token in a sentence, and a sentence is a token in a paragraph.
Now we will tokenize all the cleaned tweets in our dataset. Tokens are individual terms or words, and tokenization is the process of splitting a string of text into tokens.

During tokenization, we will also remove the following:
- Punctuation, special characters and numbers: With regards to punctuation, special characters and numbers, many, if not all, do not change or determine the overall sentiment of a tweet. Thus, it is important to remove these from the tweets.

- Short words (i.e., single letters): A vast majoity of short words which are only one letter long do not add much information surrounding the sentiment.

In [112]:
def tokenize_tweets(df):
    # tokenizing the tweets
    # Read more: https://www.kaggle.com/general/288653
    cleaned_tweets = df["message"].apply(TweetTokenizer().tokenize)

    # remove punctuation
    cleaned_tweets = cleaned_tweets.apply(lambda x : [token for token in x if token not in string.punctuation])

    # removing digits from the tweets
    cleaned_tweets = cleaned_tweets.apply(lambda x: [token for token in x if token not in list(string.digits)])

    # removing all one character tokens
    cleaned_tweets = cleaned_tweets.apply(lambda x: [token for token in x if len(token) > 1])
    
    df["tokenized_tweets"] = cleaned_tweets
    
    return df["tokenized_tweets"]

In [113]:
tokenize_tweets(df_train).head()

0    [PolySciMajor, EPA, chief, doesn't, think, car...
1    [It's, not, like, we, lack, evidence, of, anth...
2    [Researchers, say, we, have, three, years, to,...
3    [WIRED, 2016, was, pivotal, year, in, the, war...
4    [It's, 2016, and, racist, sexist, climate, cha...
Name: tokenized_tweets, dtype: object

Stop words are commonly occurring words that are often considered to be insignificant and are typically removed from text during preprocessing. These words include articles (e.g., "a," "an," "the"), prepositions (e.g., "in," "on," "at"), conjunctions (e.g., "and," "but," "or"), and other high-frequency words that do not carry substantial meaning or contribute significantly to the context of the text.

The rationale behind removing stop words is to reduce the dimensionality of the data and eliminate noise that may interfere with certain NLP tasks, such as text classification, information retrieval, and sentiment analysis. By removing these frequently occurring words, we can focus on the more informative and content-rich words that are likely to carry more significance in the analysis.

In [114]:
nltk.download('stopwords')

stop = stopwords.words('english')

# Convert to lower case
df_train["tokenized_tweets"] = df_train["tokenized_tweets"].apply(lambda x: [word.lower() for word in x])

# Remove stopwords
df_train["no_stopwords"] = df_train["tokenized_tweets"].apply(lambda x: [item for item in x if item not in stop])

df_train.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc],"[polyscimajor, epa, chief, doesn't, think, car...","[polyscimajor, epa, chief, think, carbon, diox..."
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN,"[it's, not, like, we, lack, evidence, of, anth...","[like, lack, evidence, anthropogenic, global, ..."
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]","[researchers, say, we, have, three, years, to,...","[researchers, say, three, years, act, climate,..."
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd],"[wired, 2016, was, pivotal, year, in, the, war...","[wired, 2016, pivotal, year, war, climate, cha..."
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN,"[it's, 2016, and, racist, sexist, climate, cha...","[2016, racist, sexist, climate, change, denyin..."


The words `climatechange`, `climate` and `change`, do not serve a purpose in this case since all tweets should have at least one of them. So we remove them

In [115]:
remove_words = ["climatechange", "climate", "change"]
df_train["no_climate"] = [[a for a in word if not a in remove_words] for word in df_train["no_stopwords"]]

df_train.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc],"[polyscimajor, epa, chief, doesn't, think, car...","[polyscimajor, epa, chief, think, carbon, diox...","[polyscimajor, epa, chief, think, carbon, diox..."
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN,"[it's, not, like, we, lack, evidence, of, anth...","[like, lack, evidence, anthropogenic, global, ...","[like, lack, evidence, anthropogenic, global, ..."
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]","[researchers, say, we, have, three, years, to,...","[researchers, say, three, years, act, climate,...","[researchers, say, three, years, act, late]"
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd],"[wired, 2016, was, pivotal, year, in, the, war...","[wired, 2016, pivotal, year, war, climate, cha...","[wired, 2016, pivotal, year, war]"
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN,"[it's, 2016, and, racist, sexist, climate, cha...","[2016, racist, sexist, climate, change, denyin...","[2016, racist, sexist, denying, bigot, leading..."


Create a column for each tweets sentiment

In [116]:
df_train["sentiment_name"] = df_train["sentiment"].map({-1: "Anti", 0: "Neutral", 1: "Pro", 2: "News"})
df_train.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc],"[polyscimajor, epa, chief, doesn't, think, car...","[polyscimajor, epa, chief, think, carbon, diox...","[polyscimajor, epa, chief, think, carbon, diox...",Pro
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN,"[it's, not, like, we, lack, evidence, of, anth...","[like, lack, evidence, anthropogenic, global, ...","[like, lack, evidence, anthropogenic, global, ...",Pro
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]","[researchers, say, we, have, three, years, to,...","[researchers, say, three, years, act, climate,...","[researchers, say, three, years, act, late]",News
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd],"[wired, 2016, was, pivotal, year, in, the, war...","[wired, 2016, pivotal, year, war, climate, cha...","[wired, 2016, pivotal, year, war]",Pro
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN,"[it's, 2016, and, racist, sexist, climate, cha...","[2016, racist, sexist, climate, change, denyin...","[2016, racist, sexist, denying, bigot, leading...",Pro


In [117]:
df_eda = df_train.copy()
df_eda.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc],"[polyscimajor, epa, chief, doesn't, think, car...","[polyscimajor, epa, chief, think, carbon, diox...","[polyscimajor, epa, chief, think, carbon, diox...",Pro
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN,"[it's, not, like, we, lack, evidence, of, anth...","[like, lack, evidence, anthropogenic, global, ...","[like, lack, evidence, anthropogenic, global, ...",Pro
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]","[researchers, say, we, have, three, years, to,...","[researchers, say, three, years, act, climate,...","[researchers, say, three, years, act, late]",News
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd],"[wired, 2016, was, pivotal, year, in, the, war...","[wired, 2016, pivotal, year, war, climate, cha...","[wired, 2016, pivotal, year, war]",Pro
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN,"[it's, 2016, and, racist, sexist, climate, cha...","[2016, racist, sexist, climate, change, denyin...","[2016, racist, sexist, denying, bigot, leading...",Pro


Separate dataframes of tweets for each sentiment

In [118]:
# Anti tweets
anti_df = df_train[df_train["sentiment"] == -1]
anti_df.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
28,-1,Sally Kohn’s latest evidence of climate change...,355491,NaN,[@twitchyteam],[https://t.co/mhkzogl9vt],"[sally, kohn, latest, evidence, of, climate, c...","[sally, kohn, latest, evidence, climate, chang...","[sally, kohn, latest, evidence, proves, smart,...",Anti
46,-1,Carbon Tax is a Globalist idea to enslave the...,61141,NaN,[@realdonaldtrump],NaN,"[carbon, tax, is, globalist, idea, to, enslave...","[carbon, tax, globalist, idea, enslave, world'...","[carbon, tax, globalist, idea, enslave, world'...",Anti
48,-1,: We had winds close to 100 MPH in the area t...,719523,NaN,[@stevesgoddard],NaN,"[we, had, winds, close, to, 100, mph, in, the,...","[winds, close, 100, mph, area, afternoon, woul...","[winds, close, 100, mph, area, afternoon, woul...",Anti
56,-1,lmao 😂 snowflakes ❄️ complaining about snowfl...,911385,NaN,[@misslizzynj],NaN,"[lmao, snowflakes, complaining, about, snowfla...","[lmao, snowflakes, complaining, snowflakes, wi...","[lmao, snowflakes, complaining, snowflakes, wi...",Anti
57,-1,: This is ONE of Arnold Schwarzenegger's vehi...,768263,NaN,[@dawn2334dawn],[http…],"[this, is, one, of, arnold, schwarzenegger's, ...","[one, arnold, schwarzenegger's, vehicles, whin...","[one, arnold, schwarzenegger's, vehicles, whin...",Anti


In [119]:
# Neutral tweets
neutral_df = df_train[df_train["sentiment"] == 0]
neutral_df.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
19,0,Calum: *tweets abt reunitingish w the cast*\r\...,547924,NaN,NaN,NaN,"[calum, tweets, abt, reunitingish, the, cast, ...","[calum, tweets, abt, reunitingish, cast, sees,...","[calum, tweets, abt, reunitingish, cast, sees,...",Neutral
22,0,"we also met this guy, he let us in on some tru...",67545,NaN,NaN,[https://t.co/q7yomcmzaj],"[we, also, met, this, guy, he, let, us, in, on...","[also, met, guy, let, us, truth, climate, chan...","[also, met, guy, let, us, truth, gay, people, ...",Neutral
30,0,are these the same scientists that denounce c...,365051,NaN,[@jnp_ftw],NaN,"[are, these, the, same, scientists, that, deno...","[scientists, denounce, climate, change, choice]","[scientists, denounce, choice]",Neutral
39,0,We’ ve dealt with simple issues like climate c...,403368,[#qanda],NaN,NaN,"[we, ve, dealt, with, simple, issues, like, cl...","[dealt, simple, issues, like, climate, change,...","[dealt, simple, issues, like, energy, policy, ...",Neutral
43,0,: Win probability is bullshit man. I saw the ...,326916,NaN,[@andrewsharp],NaN,"[win, probability, is, bullshit, man, saw, the...","[win, probability, bullshit, man, saw, nba, fi...","[win, probability, bullshit, man, saw, nba, fi...",Neutral


In [120]:
# Pro tweets
pro_df = df_train[df_train["sentiment"] == 1]
pro_df.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
0,1,PolySciMajor EPA chief doesn't think carbon di...,625221,NaN,[@mashable],[https://t.co/yelvcefxkc],"[polyscimajor, epa, chief, doesn't, think, car...","[polyscimajor, epa, chief, think, carbon, diox...","[polyscimajor, epa, chief, think, carbon, diox...",Pro
1,1,It's not like we lack evidence of anthropogeni...,126103,NaN,NaN,NaN,"[it's, not, like, we, lack, evidence, of, anth...","[like, lack, evidence, anthropogenic, global, ...","[like, lack, evidence, anthropogenic, global, ...",Pro
3,1,WIRED : 2016 was a pivotal year in the war on...,573736,[#todayinmaker#],NaN,[https://t.co/44wotxtlcd],"[wired, 2016, was, pivotal, year, in, the, war...","[wired, 2016, pivotal, year, war, climate, cha...","[wired, 2016, pivotal, year, war]",Pro
4,1,": It's 2016, and a racist, sexist, climate ch...",466954,[#electionnight],[@soynoviodetodas],NaN,"[it's, 2016, and, racist, sexist, climate, cha...","[2016, racist, sexist, climate, change, denyin...","[2016, racist, sexist, denying, bigot, leading...",Pro
5,1,Worth a read whether you do or don't believe i...,425577,NaN,NaN,"[https://t.co/gglzvnyjun, https://t.co/7afe2ma...","[worth, read, whether, you, do, or, don't, bel...","[worth, read, whether, believe, climate, change]","[worth, read, whether, believe]",Pro


In [121]:
# News tweets
news_df = df_train[df_train["sentiment"] == 2]
news_df.head()

,sentiment,message,tweetid,hashtags,mentions,url,tokenized_tweets,no_stopwords,no_climate,sentiment_name
2,2,: Researchers say we have three years to act ...,698562,NaN,[@rawstory],"[https://t.co/wdt0kdur2f, https://t.co/z0anpt…]","[researchers, say, we, have, three, years, to,...","[researchers, say, three, years, act, climate,...","[researchers, say, three, years, act, late]",News
12,2,: We only have a 5 percent chance of avoiding...,454673,NaN,[@tveitdal],"[https://t.co/xubtqnxhkk, https://t.co/of…]","[we, only, have, percent, chance, of, avoiding...","[percent, chance, avoiding, dangerous, global,...","[percent, chance, avoiding, dangerous, global,...",News
14,2,Fossil fuel giant ExxonMobil ‘misled’ the publ...,658092,NaN,NaN,[https://t.co/ofc2wsu4ex],"[fossil, fuel, giant, exxonmobil, misled, the,...","[fossil, fuel, giant, exxonmobil, misled, publ...","[fossil, fuel, giant, exxonmobil, misled, publ...",News
26,2,Bangladesh confronting climate change head on,365291,NaN,NaN,"[https://t.co/mtqenbqdut, https://t.co/itgkuxg...","[bangladesh, confronting, climate, change, hea...","[bangladesh, confronting, climate, change, head]","[bangladesh, confronting, head]",News
32,2,: Atmospheric rivers fueled by climate change...,143471,NaN,[@latimes],"[https://t.co/p0lzbhlu5k, https://t…]","[atmospheric, rivers, fueled, by, climate, cha...","[atmospheric, rivers, fueled, climate, change,...","[atmospheric, rivers, fueled, could, decimate,...",News
